In [7]:
# --- Import libraries ---

import os
import io
import zipfile
import warnings
import shap
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import f1_score, classification_report
from sklearn.inspection import permutation_importance
from lightgbm import LGBMClassifier
from IPython.display import display, HTML
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score, RocCurveDisplay
from imblearn.pipeline import Pipeline as ImbPipeline
from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Dropout
from tensorflow.python.keras.callbacks import EarlyStopping
warnings.filterwarnings("ignore")

In [40]:
# --- Configure pandas display options (no truncation) ---
import pandas as pd
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.max_rows', None)          # Limit rows shown, to keep output readable
pd.set_option('display.width', 0)              # Auto-detect width based on your screen
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.max_colwidth', None)    # Do not truncate long text cells

## Below is for GeneratedLabelledFlows.zip

In [ ]:
# # --- Load GeneratedLabelledFlows.zip sample ---
# zip_path_glf = "../data/GeneratedLabelledFlows.zip"
# if not os.path.exists(zip_path_glf):
#     print(f"⚠️ GeneratedLabelledFlows.zip not found at {zip_path_glf}; skipping sample load.")
# else:
#     print(f"📦 Found GeneratedLabelledFlows.zip at: {zip_path_glf}")
#     with zipfile.ZipFile(zip_path_glf, 'r') as z:
#         file_list = z.namelist()
#         print(f"\n📂 Files inside GeneratedLabelledFlows.zip:\n{file_list}\n")
#         dataframes_glf = {}
#         for filename in file_list:
#             if filename.endswith('.csv'):
#                 print(f"➡️  Loading {filename} ...")
#                 with z.open(filename) as f:
#                     try:
#                         df = pd.read_csv(f, low_memory=False, encoding='utf-8')
#                     except UnicodeDecodeError:
#                         f.seek(0)
#                         df = pd.read_csv(f, low_memory=False, encoding='latin-1')
#                     dataframes_glf[filename] = df
#                     print(f"   ✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns\n")
#         if dataframes_glf:
#             first_file = next(iter(dataframes_glf))
#             print(f"🔍 Preview of '{first_file}':")
#             display(dataframes_glf[first_file].head())


📦 Found GeneratedLabelledFlows.zip at: ../data/GeneratedLabelledFlows.zip

📂 Files inside GeneratedLabelledFlows.zip:
['TrafficLabelling /', 'TrafficLabelling /Wednesday-workingHours.pcap_ISCX.csv', 'TrafficLabelling /Tuesday-WorkingHours.pcap_ISCX.csv', 'TrafficLabelling /Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'TrafficLabelling /Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'TrafficLabelling /Monday-WorkingHours.pcap_ISCX.csv', 'TrafficLabelling /Friday-WorkingHours-Morning.pcap_ISCX.csv', 'TrafficLabelling /Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'TrafficLabelling /Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv']

➡️  Loading TrafficLabelling /Wednesday-workingHours.pcap_ISCX.csv ...
   ✅ Loaded: 692703 rows × 85 columns

➡️  Loading TrafficLabelling /Tuesday-WorkingHours.pcap_ISCX.csv ...
   ✅ Loaded: 445909 rows × 85 columns

➡️  Loading TrafficLabelling /Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv ...
   ✅ Loaded: 45896

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,192.168.10.14-209.48.71.168-49459-80-6,192.168.10.14,49459,209.48.71.168,80,6,5/7/2017 8:42,38308,1,1,6,6.0,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308.0,38308.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,20,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,192.168.10.3-192.168.10.17-389-49453-6,192.168.10.17,49453,192.168.10.3,389,6,5/7/2017 8:42,479,11,5,172,326.0,79,0,15.636364,31.449238,163,0,65.200000,89.278777,1.039666e+06,33402.922760,31.933333,25.510409,73.0,0.0,479.0,47.900000,38.942836,109.0,1.0,401.0,100.250000,101.736178,237.0,3.0,0,0,0,0,368,176,22964.509390,10438.413360,0,163,29.294118,56.529599,3195.595588,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.200000,368,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,192.168.10.3-192.168.10.17-88-46124-6,192.168.10.17,46124,192.168.10.3,88,6,5/7/2017 8:42,1095,10,6,3150,3150.0,1575,0,315.000000,632.561635,1575,0,525.000000,813.326503,5.753425e+06,14611.872150,73.000000,204.960972,810.0,1.0,1095.0,121.666667,298.746130,915.0,1.0,995.0,199.000000,345.535092,810.0,3.0,0,0,0,0,336,208,9132.420091,5479.452055,0,1575,370.588235,671.751541,451250.132400,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,336,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,192.168.10.3-192.168.10.17-389-49454-6,192.168.10.17,49454,192.168.10.3,389,6,5/7/2017 8:42,15206,17,12,3452,6660.0,1313,0,203.058823,425.778474,3069,0,555.000000,977.480342,6.650007e+05,1907.141918,543.071429,2519.931377,13391.0,0.0,15206.0,950.375000,3322.417812,13391.0,2.0,15112.0,1373.818182,4176.449588,13961.0,3.0,0,0,0,0,560,388,1117.979745,789.162173,0,3069,337.066667,704.654082,496537.374700,0,0,0,1,0,0,0,0,0,348.689655,203.058823,555.000000,560,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,192.168.10.3-192.168.10.17-88-46126-6,192.168.10.17,46126,192.168.10.3,88,6,5/7/2017 8:42,1092,9,6,3150,3152.0,1575,0,350.000000,694.509719,1576,0,525.333333,813.842901,5.771062e+06,13736.263740,78.000000,207.000929,794.0,1.0,1092.0,136.500000,313.850738,910.0,1.0,1015.0,203.000000,333.240154,794.0,3.0,0,0,0,0,304,208,8241.758242,5494.505495,0,1576,393.875000,704.585067,496440.116700,0,0,0,1,0,0,0,0,0,420.133333,350.000000,525.333333,304,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


## Below is for MachineLearningCSV.zip

In [ ]:
# # --- Load MachineLearningCSV.zip sample ---
# zip_path_glf = "../data/MachineLearningCSV.zip"
# if not os.path.exists(zip_path_glf):
#     print(f"⚠️ MachineLearningCSV.zip not found at {zip_path_glf}; skipping sample load.")
# else:
#     print(f"📦 Found MachineLearningCSV.zip at: {zip_path_glf}")
#     with zipfile.ZipFile(zip_path_glf, 'r') as z:
#         file_list = z.namelist()
#         print(f"\n📂 Files inside MachineLearningCSV.zip:\n{file_list}\n")
#         dataframes_glf = {}
#         for filename in file_list:
#             if filename.endswith('.csv'):
#                 print(f"➡️  Loading {filename} ...")
#                 with z.open(filename) as f:
#                     try:
#                         df = pd.read_csv(f, low_memory=False, encoding='utf-8')
#                     except UnicodeDecodeError:
#                         f.seek(0)
#                         df = pd.read_csv(f, low_memory=False, encoding='latin-1')
#                     dataframes_glf[filename] = df
#                     print(f"   ✅ Loaded: {df.shape[0]} rows × {df.shape[1]} columns\n")
#         if dataframes_glf:
#             first_file = next(iter(dataframes_glf))
#             print(f"🔍 Preview of '{first_file}':")
#             display(dataframes_glf[first_file].head())

📦 Found MachineLearningCSV.zip at: ../data/MachineLearningCSV.zip

📂 Files inside MachineLearningCSV.zip:
['MachineLearningCVE/', 'MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv', 'MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv', 'MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv', 'MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv', 'MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv']

➡️  Loading MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv ...
   ✅ Loaded: 692703 rows × 79 columns

➡️  Loading MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv ...
   ✅ Loaded: 445909 rows × 79 columns

➡️  Loading MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv ...
   ✅ Loaded: 17036

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308,38308,0,0.000000,0.000000,0,0,0,0.000000,0.000000,0,0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,20,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,163,0,65.200000,89.278777,1.039666e+06,33402.922760,31.933333,25.510409,73,0,479,47.900000,38.942836,109,1,401,100.250000,101.736178,237,3,0,0,0,0,368,176,22964.509390,10438.413360,0,163,29.294118,56.529599,3195.595588,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.200000,368,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,1575,0,525.000000,813.326503,5.753425e+06,14611.872150,73.000000,204.960972,810,1,1095,121.666667,298.746130,915,1,995,199.000000,345.535092,810,3,0,0,0,0,336,208,9132.420091,5479.452055,0,1575,370.588235,671.751541,451250.132400,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,336,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,3069,0,555.000000,977.480342,6.650007e+05,1907.141918,543.071429,2519.931377,13391,0,15206,950.375000,3322.417812,13391,2,15112,1373.818182,4176.449588,13961,3,0,0,0,0,560,388,1117.979745,789.162173,0,3069,337.066667,704.654082,496537.374700,0,0,0,1,0,0,0,0,0,348.689655,203.058823,555.000000,560,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,0,525.333333,813.842901,5.771062e+06,13736.263740,78.000000,207.000929,794,1,1092,136.500000,313.850738,910,1,1015,203.000000,333.240154,794,3,0,0,0,0,304,208,8241.758242,5494.505495,0,1576,393.875000,704.585067,496440.116700,0,0,0,1,0,0,0,0,0,420.133333,350.000000,525.333333,304,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [ ]:

# zip_path = "../data/MachineLearningCSV.zip"
# output_csv = "../data/FullData.csv"

# print(f"Reading all CSVs from: {zip_path}")
# print(f"Output will be saved to: {output_csv}\n")

# dfs = []
# with zipfile.ZipFile(zip_path, "r") as z:
#     csv_files = [f for f in z.namelist() if f.endswith(".csv")]
#     print(f"Found {len(csv_files)} CSV files\n")
    
#     for fname in csv_files:
#         print(f"Reading: {fname}")
#         with z.open(fname) as f:
#             try:
#                 df = pd.read_csv(f, low_memory=False, encoding="utf-8")
#             except:
#                 f.seek(0)
#                 df = pd.read_csv(f, low_memory=False, encoding="latin-1")
#         dfs.append(df)
#         print(f"  ✓ Loaded {len(df)} rows × {len(df.columns)} columns\n")

# if dfs:
#     combined = pd.concat(dfs, ignore_index=True)
#     # remove heading/trailing whitespace from column names
#     combined.columns = combined.columns.str.strip()
#     # remove Destination Port column to avoid overfitting
#     combined = combined.drop(columns=['Destination Port'])
#     # # convert Label in  "../data/FullData.csv", to TRUSTED if Label = BENIGN else UNTRUSTED 
#     combined['Label'] = combined['Label'].apply(lambda x: 'TRUSTED' if x == 'BENIGN' else 'UNTRUSTED')
#     combined.to_csv(output_csv, index=False)
#     print(f"SUCCESS: Combined all CSVs into {output_csv}")
#     print(f"Total rows: {len(combined)}")
#     print(f"Total columns: {len(combined.columns)}")
#     print(f"\nFirst few rows:")
#     display(combined.head())
# else:
#     print("[ERROR] No CSV files found in ZIP")

Reading all CSVs from: ../data/MachineLearningCSV.zip
Output will be saved to: ../data/FullData.csv

Found 8 CSV files

Reading: MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv
  ✓ Loaded 692703 rows × 79 columns

Reading: MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv
  ✓ Loaded 445909 rows × 79 columns

Reading: MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  ✓ Loaded 170366 rows × 79 columns

Reading: MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  ✓ Loaded 288602 rows × 79 columns

Reading: MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv
  ✓ Loaded 529918 rows × 79 columns

Reading: MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv
  ✓ Loaded 191033 rows × 79 columns

Reading: MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  ✓ Loaded 286467 rows × 79 columns

Reading: MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  ✓ Loaded 225745 rows × 79 colu

,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,38308,1,1,6,6,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308,38308,0,0.000000,0.000000,0,0,0,0.000000,0.000000,0,0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,20,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
1,479,11,5,172,326,79,0,15.636364,31.449238,163,0,65.200000,89.278777,1.039666e+06,33402.922760,31.933333,25.510409,73,0,479,47.900000,38.942836,109,1,401,100.250000,101.736178,237,3,0,0,0,0,368,176,22964.509390,10438.413360,0,163,29.294118,56.529599,3195.595588,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.200000,368,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
2,1095,10,6,3150,3150,1575,0,315.000000,632.561635,1575,0,525.000000,813.326503,5.753425e+06,14611.872150,73.000000,204.960972,810,1,1095,121.666667,298.746130,915,1,995,199.000000,345.535092,810,3,0,0,0,0,336,208,9132.420091,5479.452055,0,1575,370.588235,671.751541,451250.132400,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,336,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
3,15206,17,12,3452,6660,1313,0,203.058823,425.778474,3069,0,555.000000,977.480342,6.650007e+05,1907.141918,543.071429,2519.931377,13391,0,15206,950.375000,3322.417812,13391,2,15112,1373.818182,4176.449588,13961,3,0,0,0,0,560,388,1117.979745,789.162173,0,3069,337.066667,704.654082,496537.374700,0,0,0,1,0,0,0,0,0,348.689655,203.058823,555.000000,560,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
4,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,0,525.333333,813.842901,5.771062e+06,13736.263740,78.000000,207.000929,794,1,1092,136.500000,313.850738,910,1,1015,203.000000,333.240154,794,3,0,0,0,0,304,208,8241.758242,5494.505495,0,1576,393.875000,704.585067,496440.116700,0,0,0,1,0,0,0,0,0,420.133333,350.000000,525.333333,304,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED


In [41]:

csv_path = "../data/FullData.csv"
if not os.path.exists(csv_path):
  raise FileNotFoundError(f"File not found: {csv_path}")

##### Load and normalize column names

In [ ]:
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows × {len(df.columns)} columns from {csv_path}")
display(df.head())

Loaded 2830743 rows × 78 columns from ../data/FullData.csv


,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Header Length.1,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,38308,1,1,6,6,6,6,6.000000,0.000000,6,6,6.000000,0.000000,3.132505e+02,52.208416,38308.000000,0.000000,38308,38308,0,0.000000,0.000000,0,0,0,0.000000,0.000000,0,0,0,0,0,0,20,20,26.104208,26.104208,6,6,6.000000,0.000000,0.000000,0,0,0,0,1,1,0,0,1,9.000000,6.000000,6.000000,20,0,0,0,0,0,0,1,6,1,6,255,946,0,20,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
1,479,11,5,172,326,79,0,15.636364,31.449238,163,0,65.200000,89.278777,1.039666e+06,33402.922760,31.933333,25.510409,73,0,479,47.900000,38.942836,109,1,401,100.250000,101.736178,237,3,0,0,0,0,368,176,22964.509390,10438.413360,0,163,29.294118,56.529599,3195.595588,0,0,0,1,0,0,0,0,0,31.125000,15.636364,65.200000,368,0,0,0,0,0,0,11,172,5,326,29200,260,4,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
2,1095,10,6,3150,3150,1575,0,315.000000,632.561635,1575,0,525.000000,813.326503,5.753425e+06,14611.872150,73.000000,204.960972,810,1,1095,121.666667,298.746130,915,1,995,199.000000,345.535092,810,3,0,0,0,0,336,208,9132.420091,5479.452055,0,1575,370.588235,671.751541,451250.132400,0,0,0,1,0,0,0,0,0,393.750000,315.000000,525.000000,336,0,0,0,0,0,0,10,3150,6,3150,29200,2081,3,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
3,15206,17,12,3452,6660,1313,0,203.058823,425.778474,3069,0,555.000000,977.480342,6.650007e+05,1907.141918,543.071429,2519.931377,13391,0,15206,950.375000,3322.417812,13391,2,15112,1373.818182,4176.449588,13961,3,0,0,0,0,560,388,1117.979745,789.162173,0,3069,337.066667,704.654082,496537.374700,0,0,0,1,0,0,0,0,0,348.689655,203.058823,555.000000,560,0,0,0,0,0,0,17,3452,12,6660,29200,0,10,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED
4,1092,9,6,3150,3152,1575,0,350.000000,694.509719,1576,0,525.333333,813.842901,5.771062e+06,13736.263740,78.000000,207.000929,794,1,1092,136.500000,313.850738,910,1,1015,203.000000,333.240154,794,3,0,0,0,0,304,208,8241.758242,5494.505495,0,1576,393.875000,704.585067,496440.116700,0,0,0,1,0,0,0,0,0,420.133333,350.000000,525.333333,304,0,0,0,0,0,0,9,3150,6,3152,29200,2081,2,32,0.0,0.0,0,0,0.0,0.0,0,0,TRUSTED


#### Check class distribution from Label column

In [43]:
print(df['Label'].value_counts().to_frame('Count').assign(Percentage=lambda x: x['Count']/len(df)*100))
# moderate to severe class imbalance
# Consequence: A model could learn a trivial rule: always predict 1 → no feature matters. We will address this.

           Count    Percentage
Label                         
TRUSTED    2273097  80.300366 
UNTRUSTED   557646  19.699634 


#### Missing and Infinite value Analysis ####

In [44]:
# quick check for missing values
print("Any missing values in dataset?:", df.isnull().values.any())

# initial missing summary
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False)

if missing_summary.empty:
    print("No missing values found.")
else:
    # compute min/max/mean for columns with missing values (numeric only; NaN for non-numeric)
    mins = []
    maxs = []
    means = []
    for col in missing_summary.index:
        if pd.api.types.is_numeric_dtype(df[col]):
            mins.append(df[col].min(skipna=True))
            maxs.append(df[col].max(skipna=True))
            means.append(df[col].mean(skipna=True))
        else:
            mins.append(np.nan)
            maxs.append(np.nan)
            means.append(np.nan)

    missing_summary["min"] = mins
    missing_summary["max"] = maxs
    missing_summary["mean"] = means

    print("Initial missing values per column (count, percent, min, max, mean):")
    display(missing_summary)

    initial_total_missing = int(missing_counts.sum())
    print("Initial total missing cells:", initial_total_missing)


    # --- Check for infinite values in numeric columns ---
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = df[numeric_cols].apply(lambda s: int(np.isinf(s).sum()))
    inf_counts = inf_counts[inf_counts > 0].sort_values(ascending=False)

    if inf_counts.empty:
        print("No infinite (inf / -inf) values found.")
    else:
        print("Infinite values per numeric column:")
        display(inf_counts.rename("inf_count").to_frame())

        # # show example rows that contain any infinite value (up to 10)
        # rows_with_inf = np.isinf(df[numeric_cols]).any(axis=1)
        # print("\nExample rows with any infinite value (up to 10):")
        # display(df.loc[rows_with_inf].head(10))

        # replace infinite values with NaN to avoid downstream issues
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        print("Replaced inf/-inf with NaN in dataframe.")

        # update missing summary after replacement
        missing_counts_after = df.isnull().sum()
        missing_pct_after = (missing_counts_after / len(df)) * 100
        missing_summary_after = pd.DataFrame({
            "missing_count": missing_counts_after,
            "missing_pct": missing_pct_after
        })
        missing_summary_after = missing_summary_after[missing_summary_after["missing_count"] > 0].sort_values("missing_count", ascending=False)

        # recompute min/max/mean for updated missing_summary
        mins = []
        maxs = []
        means = []
        for col in missing_summary_after.index:
            if pd.api.types.is_numeric_dtype(df[col]):
                mins.append(df[col].min(skipna=True))
                maxs.append(df[col].max(skipna=True))
                means.append(df[col].mean(skipna=True))
            else:
                mins.append(np.nan)
                maxs.append(np.nan)
                means.append(np.nan)

        missing_summary_after["min"] = mins
        missing_summary_after["max"] = maxs
        missing_summary_after["mean"] = means

        print("Missing values per column AFTER replacing inf with NaN (count, percent, min, max, mean):")
        display(missing_summary_after)

        after_total_missing = int(missing_counts_after.sum())
        print("Total missing cells AFTER replacing inf:", after_total_missing)
        print("Increase in missing cells from inf replacement:", after_total_missing - initial_total_missing)

    # # show example rows that contain any missing value (after any replacement)
    # print("\nExample rows with missing values (up to 10):")
    # display(df[df.isnull().any(axis=1)].head(10))

print("Final total missing cells reported:", int(df.isnull().sum().sum()))


Any missing values in dataset?: True
Initial missing values per column (count, percent, min, max, mean):


,missing_count,missing_pct,min,max,mean
Flow Bytes/s,1358,0.047973,-261000000.0,inf,inf


Initial total missing cells: 1358
Infinite values per numeric column:


,inf_count
Flow Packets/s,2867
Flow Bytes/s,1509


Replaced inf/-inf with NaN in dataframe.
Missing values per column AFTER replacing inf with NaN (count, percent, min, max, mean):


,missing_count,missing_pct,min,max,mean
Flow Bytes/s,2867,0.101281,-261000000.0,2.071000e+09,1.491719e+06
Flow Packets/s,2867,0.101281,-2000000.0,4.000000e+06,7.085423e+04


Total missing cells AFTER replacing inf: 5734
Increase in missing cells from inf replacement: 4376
Final total missing cells reported: 5734


#### Find Label column case-insensitively

In [45]:
label_col = next((c for c in df.columns if c.lower() == "label"), None)
if label_col is None:
  raise KeyError("Column 'Label' not found in dataset")

#### Map Label -> numeric target: TRUSTED -> 1, else 0

In [12]:
def map_label(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().upper()
    if s == "TRUSTED":
        return 1
    elif s == "UNTRUSTED":
        return 0
    return np.nan   # for any unknown label


In [ ]:

df["Label"] = df[label_col].apply(map_label)
n_missing = df["Label"].isna().sum()
if n_missing:
   print(f"Dropping {n_missing} rows with missing Label/target")
df = df[df["Label"].notna()].copy()

#### Select features and target variables

In [47]:
# X = df.select_dtypes(include=[np.number]).copy()
# if "Label" in X.columns:
X = df.drop(columns=["Label"])


y = df["Label"].astype(float)
print(f"Using {X.shape[1]} features and {len(y)} target samples")

Using 77 features and 2830743 target samples


In [48]:
df['Label'].value_counts()

Label
1    2273097
0     557646
Name: count, dtype: int64

#### Impute missing numeric values

In [18]:
imp = SimpleImputer(strategy="median")


In [ ]:

X_imputed = pd.DataFrame(imp.fit_transform(X), columns=X.columns, index=X.index)

#### Feature Importance Calculation Steps ####

In [50]:
# Prepare Features and Target
print("Original class distribution:")
print(pd.Series(y).value_counts())

Original class distribution:
Label
1.0    2273097
0.0     557646
Name: count, dtype: int64


In [51]:
# Split BEFORE SMOTE 
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [52]:
# Build pipeline with SMOTE + LGBM
pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=64,
        random_state=42
    ))
])


# Fit pipeline on TRAIN ONLY
pipeline.fit(X_train, y_train)

# Extract fitted model
model = pipeline.named_steps["model"]

# Evaluate on Test Set (never SMOTEd)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))

[LightGBM] [Info] Number of positive: 1818477, number of negative: 1818477
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.737581 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14579
[LightGBM] [Info] Number of data points in the train set: 3636954, number of used features: 69
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
              precision    recall  f1-score   support

         0.0       0.99      1.00      1.00    111529
         1.0       1.00      1.00      1.00    454620

    accuracy                           1.00    566149
   macro avg       1.00      1.00      1.00    566149
weighted avg       1.00      1.00      1.00    566149

F1-macro: 0.9976851711446726


#### FEATURE IMPORTANCE ANALYSIS

In [53]:
#  Permutation Feature Importance (PFI)

pfi = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

pfi_importance = pfi.importances_mean
pfi_rank = np.argsort(-pfi_importance)

In [54]:
# SHAP Values (TreeSHAP)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values[1] is usually the class=1 contribution
if isinstance(shap_values, list):
    class_shap = shap_values[1]
else:
    class_shap = shap_values

# Mean absolute SHAP value per feature
shap_importance = np.abs(class_shap).mean(axis=0)
shap_rank = np.argsort(-shap_importance)

In [55]:
# Merge Both Rankings (Robust Importance)

feature_names = X_imputed.columns

df_importance = pd.DataFrame({
    "Feature": feature_names,
    "PFI": pfi_importance,
    "SHAP": shap_importance
})

# Normalize importance scores for comparability
df_importance["PFI_norm"] = df_importance["PFI"] / df_importance["PFI"].sum()
df_importance["SHAP_norm"] = df_importance["SHAP"] / df_importance["SHAP"].sum()

# Combined score (equal weight)
df_importance["Combined"] = (
    0.5 * df_importance["PFI_norm"] +
    0.5 * df_importance["SHAP_norm"]
)

# Sort features by combined importance
df_importance = df_importance.sort_values("Combined", ascending=False)

print("\nTop combined importance features:")
print(df_importance.head(20))



Top combined importance features:
   Feature                  PFI       SHAP           PFI_norm  SHAP_norm  \
40       Packet Length Std  0.025418  170604.317338  0.025596  0.165275    
27             Bwd IAT Max  0.041749   83688.162976  0.042040  0.081074    
65  Init_Win_bytes_forward  0.081262   36256.155660  0.081829  0.035124    
22             Fwd IAT Max  0.016960   92900.544923  0.017079  0.089998    
11  Bwd Packet Length Mean  0.056223   46845.683402  0.056615  0.045382    
12   Bwd Packet Length Std  0.050545   41306.767454  0.050898  0.040016    
5    Fwd Packet Length Max  0.008976   78204.339945  0.009039  0.075761    
9    Bwd Packet Length Max  0.056527   14313.691551  0.056921  0.013867    
8    Fwd Packet Length Std  0.015371   48837.074084  0.015478  0.047311    
13            Flow Bytes/s  0.006313   54475.210138  0.006357  0.052773    
75                Idle Max  0.046288    5100.706859  0.046610  0.004941    
24           Bwd IAT Total  0.048195    2287.300568  

In [56]:
# Compute cumulative importance
df_importance['Cumulative'] = df_importance['Combined'].cumsum()

# Select features contributing to top 95% of importance
top_95_features = df_importance[df_importance['Cumulative'] <= 0.95]['Feature'].tolist()

# Optionally include the first feature above 95% threshold to reach ≥95%
if df_importance['Cumulative'].iloc[len(top_95_features)] < 0.95:
    top_95_features.append(df_importance['Feature'].iloc[len(top_95_features)])

print(f"Number of top 95% features: {len(top_95_features)}")
print("Top 95% important features:")
print(top_95_features)

Number of top 95% features: 35
Top 95% important features:
['Packet Length Std', 'Bwd IAT Max', 'Init_Win_bytes_forward', 'Fwd IAT Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Fwd Packet Length Max', 'Bwd Packet Length Max', 'Fwd Packet Length Std', 'Flow Bytes/s', 'Idle Max', 'Bwd IAT Total', 'Packet Length Variance', 'Flow IAT Std', 'Bwd IAT Mean', 'Flow IAT Max', 'Bwd Header Length', 'Average Packet Size', 'Down/Up Ratio', 'Fwd IAT Std', 'Max Packet Length', 'Bwd IAT Std', 'Total Length of Bwd Packets', 'Bwd Packets/s', 'Bwd Packet Length Min', 'Init_Win_bytes_backward', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Bwd IAT Min', 'Packet Length Mean', 'min_seg_size_forward', 'Idle Mean', 'Fwd Packet Length Mean', 'Fwd IAT Total', 'Flow Duration']


In [ ]:
# Save importance CSV
df_importance.to_csv("feature_importance_lgbm_pfi_shap.csv", index=False)
# Save top 95% cumulative importance features
top_95_df = pd.DataFrame({'Feature': top_95_features})
top_95_df.to_csv("../data/top_95_percent_features.csv", index=False)
print("\nSaved top 95% features to 'top_95_percent_features.csv'")


Saved top 95% features to 'top_95_percent_features.csv'


In [13]:
#### Create a subset dataframe with only the top 95% features plus Label ###
top_95_percent_columns = [
    "Packet Length Std",
    "Bwd IAT Max",
    "Init_Win_bytes_forward",
    "Fwd IAT Max",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Std",
    "Fwd Packet Length Max",
    "Bwd Packet Length Max",
    "Fwd Packet Length Std",
    "Flow Bytes/s",
    "Idle Max",
    "Bwd IAT Total",
    "Packet Length Variance",
    "Flow IAT Std",
    "Bwd IAT Mean",
    "Flow IAT Max",
    "Bwd Header Length",
    "Average Packet Size",
    "Down/Up Ratio",
    "Fwd IAT Std",
    "Max Packet Length",
    "Bwd IAT Std",
    "Total Length of Bwd Packets",
    "Bwd Packets/s",
    "Bwd Packet Length Min",
    "Init_Win_bytes_backward",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Bwd IAT Min",
    "Packet Length Mean",
    "min_seg_size_forward",
    "Idle Mean",
    "Fwd Packet Length Mean",
    "Fwd IAT Total",
    "Flow Duration",
]
FullData = pd.read_csv("../data/FullData.csv")
df_top_features = FullData[top_95_percent_columns + ["Label"]].copy()
print(f"Subset dataframe with top features shape: {df_top_features.shape}")
display(df_top_features.head())


Subset dataframe with top features shape: (2830743, 36)


,Packet Length Std,Bwd IAT Max,Init_Win_bytes_forward,Fwd IAT Max,Bwd Packet Length Mean,Bwd Packet Length Std,Fwd Packet Length Max,Bwd Packet Length Max,Fwd Packet Length Std,Flow Bytes/s,...,Total Backward Packets,Total Length of Fwd Packets,Bwd IAT Min,Packet Length Mean,min_seg_size_forward,Idle Mean,Fwd Packet Length Mean,Fwd IAT Total,Flow Duration,Label
0,0.000000,0,255,0,6.000000,0.000000,6,6,0.000000,3.132505e+02,...,1,6,0,6.000000,20,0.0,6.000000,0,38308,TRUSTED
1,56.529599,237,29200,109,65.200000,89.278777,79,163,31.449238,1.039666e+06,...,5,172,3,29.294118,32,0.0,15.636364,479,479,TRUSTED
2,671.751541,810,29200,915,525.000000,813.326503,1575,1575,632.561635,5.753425e+06,...,6,3150,3,370.588235,32,0.0,315.000000,1095,1095,TRUSTED
3,704.654082,13961,29200,13391,555.000000,977.480342,1313,3069,425.778474,6.650007e+05,...,12,3452,3,337.066667,32,0.0,203.058823,15206,15206,TRUSTED
4,704.585067,794,29200,910,525.333333,813.842901,1575,1576,694.509719,5.771062e+06,...,6,3150,3,393.875000,32,0.0,350.000000,1092,1092,TRUSTED


In [14]:
print(df_top_features['Label'].value_counts().to_frame('Count').assign(Percentage=lambda x: x['Count']/len(df_top_features)*100))
# moderate to severe class imbalance
# Consequence: A model could learn a trivial rule: always predict 1 → no feature matters. We will address this.

             Count  Percentage
Label                         
TRUSTED    2273097   80.300366
UNTRUSTED   557646   19.699634


In [17]:
# quick check for missing values
print("Any missing values in dataset?:", df_top_features.isnull().values.any())

# initial missing summary
missing_counts = df_top_features.isnull().sum()
missing_pct = (missing_counts / len(df_top_features)) * 100
missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_pct": missing_pct
})
missing_summary = missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False)

if missing_summary.empty:
    print("No missing values found.")
else:
    # compute min/max/mean for columns with missing values (numeric only; NaN for non-numeric)
    mins = []
    maxs = []
    means = []
    for col in missing_summary.index:
        if pd.api.types.is_numeric_dtype(df_top_features[col]):
            mins.append(df_top_features[col].min(skipna=True))
            maxs.append(df_top_features[col].max(skipna=True))
            means.append(df_top_features[col].mean(skipna=True))
        else:
            mins.append(np.nan)
            maxs.append(np.nan)
            means.append(np.nan)

    missing_summary["min"] = mins
    missing_summary["max"] = maxs
    missing_summary["mean"] = means

    print("Initial missing values per column (count, percent, min, max, mean):")
    display(missing_summary)

    initial_total_missing = int(missing_counts.sum())
    print("Initial total missing cells:", initial_total_missing)


    # --- Check for infinite values in numeric columns ---
    numeric_cols = df_top_features.select_dtypes(include=[np.number]).columns
    inf_counts = df_top_features[numeric_cols].apply(lambda s: int(np.isinf(s).sum()))
    inf_counts = inf_counts[inf_counts > 0].sort_values(ascending=False)

    if inf_counts.empty:
        print("No infinite (inf / -inf) values found.")
    else:
        print("Infinite values per numeric column:")
        display(inf_counts.rename("inf_count").to_frame())

        # # show example rows that contain any infinite value (up to 10)
        # rows_with_inf = np.isinf(df[numeric_cols]).any(axis=1)
        # print("\nExample rows with any infinite value (up to 10):")
        # display(df.loc[rows_with_inf].head(10))

        # replace infinite values with NaN to avoid downstream issues
        df_top_features.replace([np.inf, -np.inf], np.nan, inplace=True)
        print("Replaced inf/-inf with NaN in dataframe.")

        # update missing summary after replacement
        missing_counts_after = df_top_features.isnull().sum()
        missing_pct_after = (missing_counts_after / len(df_top_features)) * 100
        missing_summary_after = pd.DataFrame({
            "missing_count": missing_counts_after,
            "missing_pct": missing_pct_after
        })
        missing_summary_after = missing_summary_after[missing_summary_after["missing_count"] > 0].sort_values("missing_count", ascending=False)

        # recompute min/max/mean for updated missing_summary
        mins = []
        maxs = []
        means = []
        for col in missing_summary_after.index:
            if pd.api.types.is_numeric_dtype(df[col]):
                mins.append(df[col].min(skipna=True))
                maxs.append(df[col].max(skipna=True))
                means.append(df[col].mean(skipna=True))
            else:
                mins.append(np.nan)
                maxs.append(np.nan)
                means.append(np.nan)

        missing_summary_after["min"] = mins
        missing_summary_after["max"] = maxs
        missing_summary_after["mean"] = means

        print("Missing values per column AFTER replacing inf with NaN (count, percent, min, max, mean):")
        display(missing_summary_after)

        after_total_missing = int(missing_counts_after.sum())
        print("Total missing cells AFTER replacing inf:", after_total_missing)
        print("Increase in missing cells from inf replacement:", after_total_missing - initial_total_missing)

    # # show example rows that contain any missing value (after any replacement)
    # print("\nExample rows with missing values (up to 10):")
    # display(df[df.isnull().any(axis=1)].head(10))

print("Final total missing cells reported:", int(df_top_features.isnull().sum().sum()))


Any missing values in dataset?: True
Initial missing values per column (count, percent, min, max, mean):


,missing_count,missing_pct,min,max,mean
Flow Bytes/s,2867,0.101281,-261000000.0,2.071000e+09,1.491719e+06


Initial total missing cells: 2867
No infinite (inf / -inf) values found.
Final total missing cells reported: 2867


#### Map Label -> numeric target: TRUSTED -> 1, else 0

In [20]:
df_top_features["Label"] = df_top_features["Label"].apply(map_label)
n_missing = df_top_features["Label"].isna().sum()
if n_missing:
   print(f"Dropping {n_missing} rows with missing Label/target")
df_top_features = df_top_features[df_top_features["Label"].notna()].copy()

In [21]:

# Save cleaned subset dataframe to CSV
df_top_features.to_csv("../data/Cleaned_Data_top_95_percent_features.csv", index=False)

### Using the top 95% features, retrain and evaluate the model ###
##### No need to run above step, if satisfied with the cleaned data 

In [22]:
cleaned_data_path = "../data/Cleaned_Data_top_95_percent_features.csv"
if not os.path.exists(cleaned_data_path):
  raise FileNotFoundError(f"File not found: {cleaned_data_path}")

In [ ]:
df_cleaned = pd.read_csv(cleaned_data_path)
print(f"Loaded {len(df_cleaned)} rows × {len(df_cleaned.columns)} columns from {cleaned_data_path}")
display(df_cleaned .head())

Loaded 2830743 rows × 36 columns from ../data/Cleaned_Data_top_95_percent_features.csv


,Packet Length Std,Bwd IAT Max,Init_Win_bytes_forward,Fwd IAT Max,Bwd Packet Length Mean,Bwd Packet Length Std,Fwd Packet Length Max,Bwd Packet Length Max,Fwd Packet Length Std,Flow Bytes/s,...,Total Backward Packets,Total Length of Fwd Packets,Bwd IAT Min,Packet Length Mean,min_seg_size_forward,Idle Mean,Fwd Packet Length Mean,Fwd IAT Total,Flow Duration,Label
0,0.000000,0,255,0,6.000000,0.000000,6,6,0.000000,3.132505e+02,...,1,6,0,6.000000,20,0.0,6.000000,0,38308,1
1,56.529599,237,29200,109,65.200000,89.278777,79,163,31.449238,1.039666e+06,...,5,172,3,29.294118,32,0.0,15.636364,479,479,1
2,671.751541,810,29200,915,525.000000,813.326503,1575,1575,632.561635,5.753425e+06,...,6,3150,3,370.588235,32,0.0,315.000000,1095,1095,1
3,704.654082,13961,29200,13391,555.000000,977.480342,1313,3069,425.778474,6.650007e+05,...,12,3452,3,337.066667,32,0.0,203.058823,15206,15206,1
4,704.585067,794,29200,910,525.333333,813.842901,1575,1576,694.509719,5.771062e+06,...,6,3150,3,393.875000,32,0.0,350.000000,1092,1092,1


#### Prepare Features and Target from cleaned data ###

In [24]:
#### Extract Features and Target from Cleaned Data ###
X_cleaned = df_cleaned.drop(columns=["Label"])   
   # take care of missing values with median imputation in numeric features 
X_cleaned_imputed = pd.DataFrame(imp.fit_transform(X_cleaned), columns=X_cleaned.columns, index=X_cleaned.index)  
y_cleaned = df_cleaned["Label"].astype(float)

# Split data (subset after feature selection)
# X_cleaned is the subset of features after selection
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned_imputed, y_cleaned, test_size=0.2, stratify=y_cleaned, random_state=42
)


# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#  Apply SMOTE on TRAINING only
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print(f"Before SMOTE: {np.bincount(y_train)}")
print(f"After SMOTE: {np.bincount(y_train_bal)}")


Before SMOTE: [ 446117 1818477]
After SMOTE: [1818477 1818477]


#### Training model on cleaned data ###

In [26]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=5, max_features='sqrt')
rf_model.fit(X_train_bal, y_train_bal)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:,1]


KeyboardInterrupt: 

In [ ]:


# Train Logistic Regression

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_bal, y_train_bal)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:,1]



In [ ]:

# Train simple CNN
input_dim = X_train_bal.shape[1]

cnn_model = Sequential([
    Dense(128, input_dim=input_dim, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = cnn_model.fit(
    X_train_bal, y_train_bal,
    validation_split=0.2,
    epochs=50,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

y_prob_cnn = cnn_model.predict(X_test_scaled).ravel()
y_pred_cnn = (y_prob_cnn >= 0.5).astype(int)


#### Evaluate Models ###

In [ ]:

# Evaluate Models
def evaluate_model(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob)
    print(f"===== {name} =====")
    print(f"Accuracy: {acc:.4f}, F1-macro: {f1:.4f}, AUC: {auc:.4f}")
    print(classification_report(y_true, y_pred))
    return acc, f1, auc

metrics = {}
metrics['RandomForest'] = evaluate_model("Random Forest", y_test, y_pred_rf, y_prob_rf)
metrics['LogisticRegression'] = evaluate_model("Logistic Regression", y_test, y_pred_lr, y_prob_lr)
metrics['CNN'] = evaluate_model("CNN", y_test, y_pred_cnn, y_prob_cnn)



In [ ]:

# Plot ROC curves
plt.figure(figsize=(8,6))
RocCurveDisplay.from_predictions(y_test, y_prob_rf, name='Random Forest', color='blue')
RocCurveDisplay.from_predictions(y_test, y_prob_lr, name='Logistic Regression', color='green')
RocCurveDisplay.from_predictions(y_test, y_prob_cnn, name='CNN', color='red')
plt.title("ROC Curves Comparison")
plt.show()



In [ ]:

# Plot F1-score bar chart
models = list(metrics.keys())
f1_scores = [metrics[m][1] for m in models]

plt.figure(figsize=(6,4))
plt.bar(models, f1_scores, color=['blue','green','red'])
plt.ylabel("F1-macro Score")
plt.title("Model Comparison on F1-macro")
plt.ylim(0,1)
plt.show()
